<a href="https://colab.research.google.com/github/emgakii001/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emgakii001/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [21]:
!git clone https://github.com/emgakii001/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 149 (delta 57), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 1.86 MiB | 23.52 MiB/s, done.
Resolving deltas: 100% (57/57), done.


In [22]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship


In [23]:
!git clone https://github.com/emgakii001/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 149 (delta 57), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 1.86 MiB | 24.74 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship


## **1. Method choice and why**

I am using Logistic Regression as my modeling method for this lane.

My lane, Refresh / Content Opportunity Scoring, requires predicting whether a page should be flagged for review — a supervised, binary outcome (needs_review). Logistic Regression fits this naturally: it outputs a probability score for each page rather than a rigid yes/no split, which allows pages to be ranked by likelihood of needing review — the same ranked-queue structure I used for my Week 4 baseline.

I chose this method over more complex options (Random Forest, Gradient Boosting) for two reasons. First, it is simple and interpretable — the model's coefficients directly show how much each feature pushes the prediction up or down, which is easy to explain to a non-technical reviewer. Second, and more importantly, it gives a fair, direct comparison against my Week 4 rule-based baseline: both produce a ranked list from a small set of signals, so any improvement in Precision@K can be attributed to combining those signals more effectively — not to added model complexity. This aligns with the principle that a model should earn its complexity, not be rewarded for it by default.

In [24]:
# Section 1 — method choice setup
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

print("Method: Logistic Regression")
print("Reason: interpretable, probability-based, directly comparable to Week 4's rule-based baseline")

# Reload starter data (fresh session check)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"\nData loaded: {df.shape}")

Method: Logistic Regression
Reason: interpretable, probability-based, directly comparable to Week 4's rule-based baseline

Data loaded: (30000, 44)


## **2. Split design**

I will use a grouped split by client rather than a plain random split, because pages belonging to the same client may share client-specific patterns — such as a particular writing style, industry, or baseline engagement level — that could make prediction easier without reflecting genuinely generalizable signal.

If pages from the same client appeared in both the training and test sets, the model could learn client-specific behavior from that client's training pages and then unfairly benefit from recognizing the same client's pages in the test set. This would inflate my evaluation score, making it look more accurate than it actually is at the task that matters — predicting whether a page needs review for a client the model has never seen before.

To prevent this, I used GroupShuffleSplit with client_id as the grouping variable, ensuring every client's full set of pages lands entirely in either the training set or the test set, never both. I verified this directly by checking for any overlap between the two sets' client IDs, confirming the split is honest before training any model on top of it.

In [25]:
from sklearn.model_selection import GroupShuffleSplit

# Grouped split by client_id — keeps each client's pages entirely in train OR test
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print(f"Train set: {len(train_df)} rows, {train_df['client_id'].nunique()} clients")
print(f"Test set: {len(test_df)} rows, {test_df['client_id'].nunique()} clients")

# Verify no client appears in both sets
overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print(f"\nClients appearing in BOTH train and test (should be 0): {len(overlap)}")

Train set: 23837 rows, 25 clients
Test set: 6163 rows, 7 clients

Clients appearing in BOTH train and test (should be 0): 0


**3. Train + compare vs my baseline**

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

To evaluate whether a trained model improves on simple rule-based logic, I trained a Logistic Regression model on the same client-grouped train/test split used throughout this notebook, using the same six features validated as leakage-free in Weeks 3-4 (impressions_90d, clicks_90d, ctr, avg_position, engagement_rate, content_age_days). I then computed Precision@50 for both the trained model and my Week 4 rule-based baseline, applied to the same test set, to ensure a fair, apples-to-apples comparison.

In [26]:
# Build the target (same logic as Week 2/4: trend_direction == "down")
df["needs_review"] = (df["trend_direction"] == "down").astype(int)

# Rebuild train/test with the target included
train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

# Features: the same signals your baseline rule used, no leakage
model_features = ["impressions_90d", "clicks_90d", "ctr", "avg_position",
                   "engagement_rate", "content_age_days"]

X_train = train_df[model_features].fillna(0)
y_train = train_df["needs_review"]

X_test = test_df[model_features].fillna(0)
y_test = test_df["needs_review"]

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (23837, 6) Test: (6163, 6)


In [27]:
from sklearn.linear_model import LogisticRegression

# Train the model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

test_df["model_score"] = model.predict_proba(X_test)[:, 1]

# Precision@50 for the MODEL
def precision_at_k(dataframe, score_col, label_col, k=50):
    top_k = dataframe.sort_values(score_col, ascending=False).head(k)
    return top_k[label_col].mean()

model_precision = precision_at_k(test_df, "model_score", "needs_review", k=50)

# Precision@50 for the WEEK 4 BASELINE RULE, applied to this same test set
test_df["flag_stale"] = (test_df["freshness_tier"] == "91-180")
tier_avg_ctr_test = test_df.groupby("position_tier")["ctr"].transform("mean")
test_df["flag_low_ctr"] = (test_df["ctr"] / tier_avg_ctr_test) < 0.14
test_df["baseline_score"] = test_df["impressions_90d"] * (tier_avg_ctr_test - test_df["ctr"]).clip(lower=0)
test_df.loc[~(test_df["flag_stale"] | test_df["flag_low_ctr"]), "baseline_score"] = 0

baseline_precision = precision_at_k(test_df, "baseline_score", "needs_review", k=50)

print(f"BASELINE (Week 4 rule)  Precision@50: {baseline_precision:.3f}")
print(f"MODEL (Logistic Regression) Precision@50: {model_precision:.3f}")

BASELINE (Week 4 rule)  Precision@50: 0.460
MODEL (Logistic Regression) Precision@50: 0.740


# **Comparison table + write-up**

Results

Method	Precision@50
Week 4 Baseline (rule-based)	0.460
Logistic Regression (this week)	0.740

The trained model outperformed the rule-based baseline by a substantial margin — 0.740 vs. 0.460 Precision@50, evaluated on the same client-grouped test set. This means that of the top 50 pages the model ranked highest, 37 genuinely needed review, compared to 23 for the baseline rule.

This improvement is consistent with the core lesson from Week 4's session: a hand-written rule can only reason about one or two signals at a time using fixed thresholds, while a trained model can weigh multiple signals — impressions, clicks, CTR, position, engagement, and content age — simultaneously and learn how they interact. The baseline rule's own weaknesses (identified in Week 4's "weak picks" review, such as flagging pages on staleness alone with no supporting CTR evidence) are exactly the kind of single-signal blind spot a multi-feature model can correct for.

This result should be read as directional and decision-support, not proof that the model has learned the true cause of page decline — it has only learned a stronger statistical pattern in this March-adjacent sample, evaluated on 7 held-out clients.

In [28]:
import pandas as pd

results = pd.DataFrame({
    "Method": ["Week 4 Baseline (rule-based)", "Logistic Regression (this week)"],
    "Precision@50": [baseline_precision, model_precision]
})

print(results.to_string(index=False))

                         Method  Precision@50
   Week 4 Baseline (rule-based)          0.46
Logistic Regression (this week)          0.74


## **4. Errors and interpretation**

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*
The model relies most heavily on CTR, since it has the largest coefficient magnitude (-0.054252) among the six features used. I should note, however, that these raw coefficients are not directly comparable across features, because the features are measured on very different scales — CTR ranges roughly between 0 and 2, while impressions range into the thousands. A small coefficient on a large-scale feature can still carry real predictive weight, so this ranking should be read cautiously rather than as a definitive importance order.

Looking at the errors, the model produced 1,593 false positives (25.8% of the test set) and 1,258 false negatives (20.4%). Sample false positives tend to involve pages with low impressions, low clicks, and relatively low CTR — cases that look similar in shape to pages that genuinely do need review, which likely explains why the model leaned toward flagging them. Sample false negatives show a similar overlap: pages with low CTR and limited search exposure that the model still predicted as not needing review. This overlap in characteristics between the two error types suggests these are inherently difficult cases to separate using the current feature set alone.

Examining the full range of model scores for each error type adds an important nuance. False positive scores ranged from 0.500 to 0.796, and the small sample reviewed by hand fell in a similarly borderline range (0.52-0.59) — consistent with the model being mildly, not confidently, wrong on these. False negatives, however, ranged much more widely, from 0.005 to 0.500. While the sample rows reviewed by hand happened to show borderline scores (0.37-0.49), the full range reveals that some false negatives were cases the model was highly confident did not need review, despite the actual label showing otherwise. This means the model's errors are not uniformly a matter of borderline uncertainty — a subset represents genuine, confident misses that would be worth investigating further, for example by checking whether these pages have some unusual characteristic not captured by the current six features.

In [29]:
import numpy as np

coefficients = pd.DataFrame({
    "feature": model_features,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", ascending=False)

print("Feature coefficients (positive = pushes toward 'needs review'):")
print(coefficients.to_string(index=False))

Feature coefficients (positive = pushes toward 'needs review'):
         feature  coefficient
    avg_position     0.000006
 impressions_90d     0.000004
 engagement_rate    -0.001652
      clicks_90d    -0.002212
content_age_days    -0.003001
             ctr    -0.054252


In [30]:
test_df["predicted_label"] = (test_df["model_score"] >= 0.5).astype(int)

false_positives = test_df[(test_df["predicted_label"] == 1) & (test_df["needs_review"] == 0)]
false_negatives = test_df[(test_df["predicted_label"] == 0) & (test_df["needs_review"] == 1)]

print(f"False positives: {len(false_positives)} ({len(false_positives)/len(test_df)*100:.1f}% of test set)")
print(f"False negatives: {len(false_negatives)} ({len(false_negatives)/len(test_df)*100:.1f}% of test set)")

print("\nSample false positives (model said review, actually fine):")
print(false_positives[model_features + ["model_score"]].head(3).to_string())

print("\nSample false negatives (model missed a real review case):")
print(false_negatives[model_features + ["model_score"]].head(3).to_string())

False positives: 1593 (25.8% of test set)
False negatives: 1258 (20.4% of test set)

Sample false positives (model said review, actually fine):
    impressions_90d  clicks_90d   ctr  avg_position  engagement_rate  content_age_days  model_score
13              307           0  0.00          39.8              0.0               238     0.569934
26             2426           3  0.12          30.0              0.0               300     0.522492
36              371           5  1.35           5.4              0.0               187     0.586708

Sample false negatives (model missed a real review case):
    impressions_90d  clicks_90d   ctr  avg_position  engagement_rate  content_age_days  model_score
1             15320           7  0.05          20.3              0.0               445     0.424771
23              297           1  0.34          13.9              0.0               502     0.370143
39                4           0  0.00          36.3              0.0               348     0.4875

In [31]:
# Section 4 — feature coefficients and error analysis summary
print("Feature coefficients (raw, not standardized — see caveat in write-up):")
print(coefficients.to_string(index=False))

print(f"\nFalse positives: {len(false_positives)} ({len(false_positives)/len(test_df)*100:.1f}%)")
print(f"False negatives: {len(false_negatives)} ({len(false_negatives)/len(test_df)*100:.1f}%)")

print(f"\nFalse positive model scores — range: {false_positives['model_score'].min():.3f} to {false_positives['model_score'].max():.3f}")
print(f"False negative model scores — range: {false_negatives['model_score'].min():.3f} to {false_negatives['model_score'].max():.3f}")

Feature coefficients (raw, not standardized — see caveat in write-up):
         feature  coefficient
    avg_position     0.000006
 impressions_90d     0.000004
 engagement_rate    -0.001652
      clicks_90d    -0.002212
content_age_days    -0.003001
             ctr    -0.054252

False positives: 1593 (25.8%)
False negatives: 1258 (20.4%)

False positive model scores — range: 0.500 to 0.796
False negative model scores — range: 0.005 to 0.500


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.